In [1]:
# ==========================================================
# PUBLICATION-QUALITY DETECTION ANALYSIS
# ==========================================================
# Performs:
# 1) First-passage detection extraction
# 2) Ensemble averaging with error bars (SEM)
# 3) Detection time distribution
# 4) 1/N scaling test (log-log)
# 5) Ergodic product check
# 6) Simulation sufficiency test
#
# Works even if only subset of Np values were simulated.
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
import glob
import re
import os

In [2]:
# ==========================================================
# USER PARAMETERS
# ==========================================================

TIMESTEP_SIZE = 1e-8       # seconds
R_INTERACTION = 1.5e-6     # meters
TOTAL_PARTICLES = 100

In [3]:
# ==========================================================
# PLOT STYLE (Publication Quality)
# ==========================================================

plt.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "legend.fontsize": 12,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "lines.linewidth": 2,
    "figure.figsize": (6,5),
    "figure.dpi": 300
})

In [4]:
# ==========================================================
# MINIMUM IMAGE CONVENTION
# ==========================================================

def minimum_image(dx, box_length):
    dx -= box_length * np.round(dx / box_length)
    return dx

In [5]:
# ==========================================================
# FIRST DETECTION FUNCTION
# ==========================================================

def compute_first_detection(filename):

    detected = False
    detection_timestep = None
    last_timestep = None

    with open(filename, 'r') as f:

        while True:
            line = f.readline()
            if not line:
                break

            if "ITEM: TIMESTEP" in line:
                timestep = int(f.readline().strip())
                last_timestep = timestep

            if "ITEM: BOX BOUNDS" in line:
                x_bounds = f.readline().split()
                y_bounds = f.readline().split()
                f.readline()

                xlo, xhi = float(x_bounds[0]), float(x_bounds[1])
                ylo, yhi = float(y_bounds[0]), float(y_bounds[1])

                Lx = xhi - xlo
                Ly = yhi - ylo

            if "ITEM: ATOMS" in line:

                probes = []
                foreign = []

                for _ in range(TOTAL_PARTICLES):
                    parts = f.readline().split()
                    atom_type = int(parts[1])
                    x = float(parts[2])
                    y = float(parts[3])

                    if atom_type == 1:
                        probes.append([x, y])
                    else:
                        foreign.append([x, y])

                probes = np.array(probes)
                foreign = np.array(foreign)

                if len(probes) == 0 or len(foreign) == 0:
                    continue

                dx = probes[:, None, 0] - foreign[None, :, 0]
                dy = probes[:, None, 1] - foreign[None, :, 1]

                dx = minimum_image(dx, Lx)
                dy = minimum_image(dy, Ly)

                distances = np.sqrt(dx**2 + dy**2)
                dmin = np.min(distances)

                if not detected and dmin < R_INTERACTION:
                    detected = True
                    detection_timestep = timestep

    if detected:
        detection_time = detection_timestep * TIMESTEP_SIZE
    else:
        detection_time = None

    simulation_time = last_timestep * TIMESTEP_SIZE

    return detected, detection_time, simulation_time

# ==========================================================
# FIND ALL AVAILABLE Np VALUES
# ==========================================================

def find_available_Np():

    files = glob.glob("dump_Np*_Nm*_seed*")
    Np_set = set()

    for file in files:
        match = re.search(r"dump_Np(\d+)_Nm", file)
        if match:
            Np_set.add(int(match.group(1)))

    return sorted(list(Np_set))

In [6]:
# ==========================================================
# MAIN ANALYSIS
# ==========================================================

def analyze_all():

    Np_values = find_available_Np()

    if len(Np_values) == 0:
        print("No dump files found.")
        return

    mean_T_values = []
    sem_T_values = []
    ergodic_products = []
    ergodic_errors = []

    print("\n===== STARTING ANALYSIS =====\n")

    for Np in Np_values:

        pattern = f"dump_Np{Np}_Nm*_seed*"
        files = glob.glob(pattern)

        detection_times = []
        total_runs = len(files)

        for file in files:
            detected, T, sim_time = compute_first_detection(file)
            if detected:
                detection_times.append(T)

        detection_times = np.array(detection_times)
        detection_fraction = len(detection_times) / total_runs

        if len(detection_times) < 2:
            print(f"Np={Np} not enough detections for statistics.\n")
            continue

        mean_T = np.mean(detection_times)
        std_T  = np.std(detection_times)
        sem_T  = std_T / np.sqrt(len(detection_times))

        mean_T_values.append(mean_T)
        sem_T_values.append(sem_T)

        ergodic_products.append(Np * mean_T)
        ergodic_errors.append(Np * sem_T)

        print(f"Np = {Np}")
        print(f"  Mean detection time = {mean_T:.3e} s")
        print(f"  SEM                 = {sem_T:.3e} s")
        print(f"  Detection fraction  = {detection_fraction:.2f}")
        print("")

        # Distribution plot
        plt.figure()
        plt.hist(detection_times, bins=20, density=True)
        plt.xlabel("Detection Time (s)")
        plt.ylabel("Probability Density")
        plt.title(f"Detection Time Distribution (Np={Np})")
        plt.tight_layout()
        plt.savefig(f"distribution_Np{Np}.pdf")
        plt.close()

        if detection_fraction < 0.9:
            print(f"WARNING: Np={Np} may need longer simulation time.\n")

    # Convert to arrays
    Np_values = np.array(Np_values[:len(mean_T_values)])
    mean_T_values = np.array(mean_T_values)
    sem_T_values = np.array(sem_T_values)
    ergodic_products = np.array(ergodic_products)
    ergodic_errors = np.array(ergodic_errors)

    # ======================================================
    # 1/N SCALING PLOT
    # ======================================================

    plt.figure()
    plt.errorbar(Np_values, mean_T_values, yerr=sem_T_values, fmt='o', capsize=4)
    plt.yscale("log")
    plt.xlabel("Number of Probes (Np)")
    plt.ylabel("Mean Detection Time ⟨T⟩ (s)")
    plt.title("Scaling of Detection Time")
    plt.tight_layout()
    plt.savefig("scaling_with_errorbars.pdf")
    plt.close()
    # ======================================================
    # ERGODIC PRODUCT PLOT
    # ======================================================

    plt.figure()
    plt.errorbar(Np_values, ergodic_products,
                 yerr=ergodic_errors,
                 fmt='o', capsize=4)

    plt.xlabel("Number of Probes (Np)")
    plt.ylabel("Np × ⟨T⟩ (s)")
    plt.title("Ergodic Product Check")
    plt.tight_layout()
    plt.savefig("ergodic_product_with_errorbars.pdf")
    plt.close()

    print("\n===== ANALYSIS COMPLETE =====")
    print("Generated:")
    print(" - distribution_NpX.pdf")
    print(" - scaling_with_errorbars.pdf")
    print(" - ergodic_product_with_errorbars.pdf\n")


# ==========================================================
# RUN
# ==========================================================

if __name__ == "__main__":
    analyze_all()


===== STARTING ANALYSIS =====

Np = 2
  Mean detection time = 5.743e-01 s
  SEM                 = 6.974e-02 s
  Detection fraction  = 1.00

Np = 4
  Mean detection time = 5.727e-01 s
  SEM                 = 1.121e-01 s
  Detection fraction  = 1.00

Np = 6
  Mean detection time = 5.174e-01 s
  SEM                 = 5.720e-02 s
  Detection fraction  = 1.00

Np = 8
  Mean detection time = 3.685e-01 s
  SEM                 = 5.406e-02 s
  Detection fraction  = 1.00

Np = 60
  Mean detection time = 3.247e-01 s
  SEM                 = 1.853e-02 s
  Detection fraction  = 1.00

Np = 70
  Mean detection time = 2.806e-01 s
  SEM                 = 2.140e-02 s
  Detection fraction  = 1.00

Np = 80
  Mean detection time = 3.775e-01 s
  SEM                 = 3.272e-02 s
  Detection fraction  = 1.00

Np = 90
  Mean detection time = 3.835e-01 s
  SEM                 = 3.263e-02 s
  Detection fraction  = 1.00

Np = 99
  Mean detection time = 1.128e+00 s
  SEM                 = 1.320e-01 s
  Detection 